In [1]:
import pandas as pd
import psycopg2
from datetime import datetime

sheet_curr = '1wxR3iYna86qrdViwHjUPzHuw6bCNeMLb72M25hpUHYk'
sheet_archive = '1PxNYGMXaVrRqI0uyMQF46K7nDEG16WnDoKrFyI_qrvE'
gid_matches = '2141931777'
gid_deck = '590005429'

# MATCH_ID      = 11000000000
# EVENT_ID      = 12000000000
# DECK_ID       = 13000000000
# EVENT_TYPE_ID = 14000000000
# LOAD_RPT_ID   = 15000000000
# EV_REJ_ID     = 16000000000
# MATCH_REJ_ID  = 17000000000

In [2]:
with open("credentials.txt", "r") as file:
    credentials = [line.strip() for line in file]

In [3]:
def conn(query,vars=(), credentials=credentials):
    try:
        conn = psycopg2.connect(
            host=credentials[0],
            port=credentials[1],
            user=credentials[2],
            password=credentials[3],
            database=credentials[4]
        )
        cursor = conn.cursor()

        if len(vars) > 0:
            cursor.execute(query,vars)
        else:
            cursor.execute(query)

        conn.commit()
    except psycopg2.Error as e:
        print('Error:', e)
    finally:
        if conn:
            cursor.close()
            conn.close()

In [4]:
def parse_class_sheet(sheet,gid):
    deck_url = f'https://docs.google.com/spreadsheets/d/{sheet}/export?format=csv&gid={gid}'
    df = pd.read_csv(deck_url)

    # Create dataframe with valid Deck Names.
    df_decks = df[['Archetype','Subarchetype']].sort_values(['Archetype','Subarchetype']).reset_index(drop=True)
    df_decks.columns = ['ARCHETYPE','SUBARCHETYPE']
    df_decks['ARCHETYPE'] = df_decks['ARCHETYPE'].str.strip().str.upper()
    df_decks['SUBARCHETYPE'] = df_decks['SUBARCHETYPE'].str.strip().str.upper()

    # Adding rows for NA and NO SHOW.
    df_decks = pd.concat([df_decks,pd.DataFrame({'ARCHETYPE':['NA','NA','NA'],'SUBARCHETYPE':['NA','NO SHOW','INVALID_NAME']})],ignore_index=True)

    # Create dataframe with valid Event Types.
    df_events = df[['Event Types']]
    df_events.columns = ['EVENT_TYPE']
    df_events = df_events.dropna(subset=['EVENT_TYPE'])
    df_events['EVENT_TYPE'] = df_events['EVENT_TYPE'].str.strip().str.upper()

    # Adding row for NA.
    df_events = pd.concat([df_events,pd.DataFrame({'EVENT_TYPE':['INVALID_TYPE']})],ignore_index=True)

    # Add Format column to Decks table.
    df_decks['FORMAT'] = 'VINTAGE'
    df_decks = df_decks[['FORMAT','ARCHETYPE','SUBARCHETYPE']]

    # Add Format column to Events table.
    df_events['FORMAT'] = 'VINTAGE'
    df_events = df_events[['FORMAT','EVENT_TYPE']]
    
    return (df_decks,df_events)

In [5]:
def insert(df_valid_decks=None, df_valid_event_types=None, credentials=credentials):
    valid_decks_query = """
        INSERT INTO "VALID_DECKS" ("FORMAT", "ARCHETYPE", "SUBARCHETYPE", "PROC_DT")
        VALUES (%s, %s, %s, %s)
        ON CONFLICT ("FORMAT", "ARCHETYPE", "SUBARCHETYPE")
        DO NOTHING
    """
    valid_event_types_query = """
        INSERT INTO "VALID_EVENT_TYPES" ("FORMAT", "EVENT_TYPE", "PROC_DT")
        VALUES (%s, %s, %s)
        ON CONFLICT ("FORMAT", "EVENT_TYPE") 
        DO NOTHING
    """
    proc_dt = datetime.now()
    try:
        conn = psycopg2.connect(
            host=credentials[0],
            port=credentials[1],
            user=credentials[2],
            password=credentials[3],
            database=credentials[4]
        )
        cursor = conn.cursor()

        # Insert valid_decks
        if df_valid_decks is not None:
            values_list = [(row['FORMAT'], row['ARCHETYPE'], row['SUBARCHETYPE'], proc_dt) for index, row in df_valid_decks.iterrows()]
            for values in values_list:
                try:
                    print(values)
                    cursor.execute(valid_decks_query, values)
                except Exception as e:
                    print(f"Error inserting row into VALID_DECKS: {values} | Error: {e}")
                    continue  # Skip the row and continue with the next one
            conn.commit()

        # Insert valid_event_types
        if df_valid_event_types is not None:
            values_list = [(row['FORMAT'], row['EVENT_TYPE'], proc_dt) for index, row in df_valid_event_types.iterrows()]
            for values in values_list:
                try:
                    print(values)
                    cursor.execute(valid_event_types_query, values)
                except Exception as e:
                    print(f"Error inserting row into VALID_EVENT_TYPES: {values} | Error: {e}")
                    continue
            conn.commit()

    except Exception as e:
        print(f"Error occurred while loading data: {e}")
        conn.rollback()

    finally:
        if conn:
            cursor.close()
            conn.close()

In [6]:
def delete_table(TABLE):
    query = f'DROP TABLE IF EXISTS "{TABLE}" CASCADE'
    conn(query)

In [7]:
df_valid_decks, df_valid_event_types = parse_class_sheet(sheet_curr,gid_deck)
insert(df_valid_decks,df_valid_event_types)

('VINTAGE', 'AGGRO', 'ELDRAZI', datetime.datetime(2025, 2, 13, 18, 58, 21, 924846))
('VINTAGE', 'AGGRO', 'INITIATIVE', datetime.datetime(2025, 2, 13, 18, 58, 21, 924846))
('VINTAGE', 'AGGRO', 'MERFOLK', datetime.datetime(2025, 2, 13, 18, 58, 21, 924846))
('VINTAGE', 'AGGRO', 'OTHER AGGRO', datetime.datetime(2025, 2, 13, 18, 58, 21, 924846))
('VINTAGE', 'AGGRO', 'RED PRISON', datetime.datetime(2025, 2, 13, 18, 58, 21, 924846))
('VINTAGE', 'BAZAAR', 'AGGROVINE', datetime.datetime(2025, 2, 13, 18, 58, 21, 924846))
('VINTAGE', 'BAZAAR', 'COUNTERVINE', datetime.datetime(2025, 2, 13, 18, 58, 21, 924846))
('VINTAGE', 'BAZAAR', 'DREDGE', datetime.datetime(2025, 2, 13, 18, 58, 21, 924846))
('VINTAGE', 'COMBO', 'BESEECH STORM', datetime.datetime(2025, 2, 13, 18, 58, 21, 924846))
('VINTAGE', 'COMBO', 'BREACH', datetime.datetime(2025, 2, 13, 18, 58, 21, 924846))
('VINTAGE', 'COMBO', 'DOOMSDAY', datetime.datetime(2025, 2, 13, 18, 58, 21, 924846))
('VINTAGE', 'COMBO', 'OOPS ALL SPELLS', datetime.dat

### Test functions.

In [8]:
len(df_valid_decks)

33

In [9]:
len(df_valid_event_types)

6

In [10]:
#df_valid_decks.iloc[10,2] = 'POOPSDAY2'
#df_valid_event_types.iloc[2,1] = 'POOPLIM2'

In [11]:
#insert(df_valid_decks,df_valid_event_types)